In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark


spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[1]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [6]:
df_categories =df_read_business_data.select(F.col("categories"))

In [9]:
def get_category_name(text):

    if text is None:
       return "Not Specified", ["Not Specified"]
    
    list_of_element = text.split(",")
    category_name=list_of_element[0].strip()
    del list_of_element[0]
    if len(list_of_element) == 0:
        return category_name, ["Not Specified"]

    list_of_element_without_space=[st.lower().strip() for st in list_of_element]

    return category_name,list_of_element_without_space  

schema = StructType([
    StructField("Categorie_name", StringType(), True),
    StructField("tag_name", ArrayType(StringType()), True),  # <-- fixed
])

get_category_name_udf = F.udf(get_category_name, schema)

df_elite = (
    df_categories.withColumn("new_eilte", get_category_name_udf(F.col("categories")))
     .withColumn("Categorie_name", F.col("new_eilte.Categorie_name"))
     .withColumn("tag_name", F.explode(F.col("new_eilte.tag_name")))
).drop(F.col("new_eilte"),F.col("categories"))

df_elite_3 = df_elite.withColumn("Subcategory_pk_sk", F.monotonically_increasing_id() + 1)




In [26]:
df_elite_3.show()

+-----------------+--------------------+-----------------+
|   Categorie_name|            tag_name|Subcategory_pk_sk|
+-----------------+--------------------+-----------------+
|          Doctors|traditional chine...|                1|
|          Doctors|naturopathic/holi...|                2|
|          Doctors|         acupuncture|                3|
|          Doctors|    health & medical|                4|
|          Doctors|       nutritionists|                5|
| Shipping Centers|      local services|                6|
| Shipping Centers|            notaries|                7|
| Shipping Centers|     mailbox centers|                8|
| Shipping Centers|   printing services|                9|
|Department Stores|            shopping|               10|
|Department Stores|             fashion|               11|
|Department Stores|       home & garden|               12|
|Department Stores|         electronics|               13|
|Department Stores|    furniture stores|               1

In [30]:
df_read_dim_category=spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")

In [32]:
df_read_dim_category.columns

['Category_name', 'Category_pk_sk']

In [33]:
df_elite_3.join(df_read_dim_category, df_elite_3.Categorie_name == df_read_dim_category.Category_name, how="inner").show()



Py4JJavaError: An error occurred while calling o265.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 16.0 failed 1 times, most recent failure: Lost task 0.0 in stage 16.0 (TID 16) (amr executor driver): TaskResultLost (result lost from block manager)
Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$3(DAGScheduler.scala:3318)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:3318)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:3310)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:3310)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:1363)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DAGScheduler.scala:1363)

In [ ]:
dict_longitude =dict()

for i,j in df_read_dim_category.collect():
   dict_longitude[i] = j





In [ ]:
# Declare an empty Dictionary
dict = {}

# Convert PySpark DataFrame to Pandas 
# DataFrame
df = df_read_dim_category.toPandas()

# Traverse through each column
for column in df.columns:

    # Add key as column_name and
    # value as list of column values
    dict["Category_name"] = df["Category_pk_sk"].values.tolist()

# Print the dictionary
print(dict)

In [21]:

from pyspark.sql.functions import create_map, col

# Initialize Spark


# Combine two columns into a key-value map column
df_with_map = df_read_dim_category.withColumn("Category_pk_sk", create_map(col("Category_name"), col("Category_pk_sk")))

df_with_map.show(truncate=False)


+-------------------------+---------------------------------+
|Category_name            |Category_pk_sk                   |
+-------------------------+---------------------------------+
|Doctors                  |{Doctors -> 1}                   |
|Shipping Centers         |{Shipping Centers -> 2}          |
|Department Stores        |{Department Stores -> 3}         |
|Restaurants              |{Restaurants -> 4}               |
|Brewpubs                 |{Brewpubs -> 5}                  |
|Burgers                  |{Burgers -> 6}                   |
|Sporting Goods           |{Sporting Goods -> 7}            |
|Synagogues               |{Synagogues -> 8}                |
|Pubs                     |{Pubs -> 9}                      |
|Ice Cream & Frozen Yogurt|{Ice Cream & Frozen Yogurt -> 10}|
|Department Stores        |{Department Stores -> 11}        |
|Vietnamese               |{Vietnamese -> 12}               |
|American (Traditional)   |{American (Traditional) -> 13}   |
|General

In [ ]:
map_expr = F.create_map( [F.lit(x) for x in chain(*dict_longitude.items())] )

df_elite_4 =  df_elite_3.withColumn("Category_pk_sk" , map_expr.getItem(F.col("Categorie_name")))

In [ ]:
collected_data = df_with_map.collect()
dictionary = {row["Category_name"]: row["Category_pk_sk"] for row in collected_data}

print("Dictionary:", dictionary)
print("Type:", type(dictionary))

TypeError: cannot use 'pyspark.sql.classic.column.Column' as a dict key (unhashable type: 'Column')

In [ ]:
df_read_dim_category.columns

['Category_name', 'Category_pk_sk']

In [17]:
df_read_dim_category.show()

+--------------------+--------------+
|       Category_name|Category_pk_sk|
+--------------------+--------------+
|             Doctors|             1|
|    Shipping Centers|             2|
|   Department Stores|             3|
|         Restaurants|             4|
|            Brewpubs|             5|
|             Burgers|             6|
|      Sporting Goods|             7|
|          Synagogues|             8|
|                Pubs|             9|
|Ice Cream & Froze...|            10|
|   Department Stores|            11|
|          Vietnamese|            12|
|American (Traditi...|            13|
|   General Dentistry|            14|
|                Food|            15|
|          Sushi Bars|            16|
|          Automotive|            17|
|          Vape Shops|            18|
|          Automotive|            19|
|              Korean|            20|
+--------------------+--------------+
only showing top 20 rows
